In [ ]:
import os
import sys
import json
from pathlib import Path

from ipywidgets import widgets
from ipyfilechooser import FileChooser
from IPython.display import display, clear_output

sys.path.append(os.path.abspath('..'))
from care.l5 import do_data_processing_l5, test_loss_l5, do_training_l5
from care.data_augmentation import apply_data_augmentation_to_npz

In [ ]:
start_dir = os.path.abspath('../../data') if os.path.isdir('../../data') else os.getcwd()
fc_data = FileChooser(start_dir)
fc_data.title = '<b>Select the "Training_L5" folder (Low, High, Mask):</b>'
fc_data.show_only_dirs = True

w_t_low = widgets.IntText(value=3, description='Time window for low images:', style={'description_width': 'initial'})
w_val = widgets.FloatSlider(value=0.1, min=0.05, max=0.15, step=0.01, description='Validation fraction:', style={'description_width': 'initial'})
btn_npz = widgets.Button(description='Generate .npz', button_style='success', icon='play')
out_npz = widgets.Output()

def on_npz_clicked(b):
    global DATASET_DIR, MODELS_ROOT, NPZ_TRAIN, NPZ_VAL, DATASET_INFO
    with out_npz:
        clear_output()
        if fc_data.selected is None:
            print("Error: please select the dataset folder.")
            return
        DATASET_DIR = Path(fc_data.selected)
        MODELS_ROOT = DATASET_DIR.parent / 'models'
        NPZ_TRAIN, NPZ_VAL = do_data_processing_l5(DATASET_DIR, MODELS_ROOT / f"{DATASET_DIR.parent.name}_L5",
                                                   w_t_low.value, w_val.value)
        info_file = DATASET_DIR / 'dataset_info.json'
        DATASET_INFO = json.load(open(info_file)) if info_file.exists() else {}
        print("Mask size:", DATASET_INFO.get('mask_size'), "| detection threshold:", DATASET_INFO.get('threshold'))

btn_npz.on_click(on_npz_clicked)
display(widgets.VBox([fc_data, w_t_low, w_val, btn_npz, out_npz]))

In [ ]:
test_loss_l5(w=10, lam=0)
test_loss_l5(w=10, lam=10)

In [ ]:
w_kernel = widgets.IntText(value=3, description='Kernel Size:')
w_depth = widgets.IntText(value=2, description='Net Depth:')
w_first = widgets.IntText(value=64, description='First Layer:')
w_epochs = widgets.IntText(value=100, description='Epochs:')
w_lr = widgets.FloatText(value=1e-4, description='Learn Rate:', step=1e-5)
w_w = widgets.FloatText(value=10.0, description='w (molecules):')
w_lam = widgets.FloatText(value=0.0, description='λ (invented):')
w_thr = widgets.FloatText(value=DATASET_INFO.get('threshold', 200.0), description='Threshold:')
w_ms = widgets.IntText(value=DATASET_INFO.get('mask_size', 7), description='Mask size:')
w_seed = widgets.IntText(value=0, description='Seed:')

display(widgets.VBox([
    widgets.HTML("<b>Architecture</b>"), widgets.HBox([w_kernel, w_depth, w_first]),
    widgets.HTML("<b>Training</b>"), widgets.HBox([w_epochs, w_lr, w_seed]),
    widgets.HTML("<b>L5 loss</b> (w = 0 and λ = 0: standard MAE)"), widgets.HBox([w_w, w_lam]), widgets.HBox([w_thr, w_ms]),
]))

In [ ]:
do_training_l5(NPZ_TRAIN, NPZ_VAL, MODELS_ROOT,
               kernel_size=w_kernel.value, unet_depth=w_depth.value, unet_first_layer=w_first.value,
               train_epochs=30, lr=w_lr.value,
               w=10, lam=10, w_ghost=10, threshold=w_thr.value, mask_size=w_ms.value,
               seed=w_seed.value, dataset_dir=DATASET_DIR)

In [ ]:
import json
import numpy as np
import tensorflow as tf
from scipy import ndimage as ndi
from csbdeep.models import CARE
from csbdeep.io import load_training_data
from care.l5 import _spots, mae_img, mae_mol, mae_ghost

(X_val, Y_val), _, _ = load_training_data(NPZ_VAL, validation_split=0, verbose=False)
yt = tf.constant(Y_val)
masks = (Y_val[..., 1] == 1).astype(np.float32) 

for d in sorted(MODELS_ROOT.glob('L5_*')):
    info = json.load(open(d / 'run_info.json'))
    model = CARE(None, d.name, basedir=str(MODELS_ROOT))          # loads weights_best.h5
    pred = tf.constant(model.keras_model.predict(X_val, batch_size=16, verbose=0))
    spots = _spots(pred, w_thr.value).numpy()[..., 0]
    invented = (spots * (1 - masks)).sum() / len(X_val)
    found = total = 0
    for s, m in zip(spots, masks):
        lab, n = ndi.label(m)
        total += n
        found += len(set(np.unique(lab[s > 0])) - {0})
    print(f"{d.name:36s} ({info['data_train']['n_patches']} patches)  mae_img {float(mae_img(yt, pred)):7.2f}  "
          f"mae_mol {float(mae_mol(yt, pred)):7.2f}  mae_ghost {float(mae_ghost(yt, pred)):7.2f}  "
          f"invented/patch {invented:.3f}  molecules found {100 * found / total:.0f} %")